In [ ]:
# General notebook settings
import logging
import warnings

import pypsa

warnings.filterwarnings("error", category=DeprecationWarning)
# pandas<3.0.3 sets the `locs` attribute deprecated in matplotlib>=3.11
warnings.filterwarnings("ignore", message="The locs attribute was deprecated")
logging.getLogger("gurobipy").propagate = False
pypsa.options.params.optimize.log_to_console = False

# Modular Expansion

In this example, we demonstrate how to handle the modular expansion feature in PyPSA. Modular expansion allows you to specify discrete steps for the expansion of components. This is particularly useful for technologies that can only be built in fixed block sizes, such as nuclear generators.

We start by loading the 3-hourly resolved [model.energy example](./capacity-expansion-planning-single-node.ipynb) and adding a nuclear generator as expansion option (with ramp limits of 3%/h and a minimum part load of 70%). Initially, we allow the nuclear generator to be built in continuous sizes.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pypsa
from pypsa.costs import annuity

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})

n = pypsa.examples.model_energy()

n.add(
    "Generator",
    "nuclear",
    bus="electricity",
    p_nom_extendable=True,
    marginal_cost=15,
    capital_cost=annuity(0.07, 50) * 8_000_000,
    p_min_pu=0.7,
    ramp_limit_up=0.03,
    ramp_limit_down=0.03,
)

n.optimize(solver_name="gurobi")

p_nom_continuous = n.generators.p_nom_opt.copy()
capex_continuous = n.statistics.capex(components="Generator", groupby="name")

As optimal capacity for the nuclear generator, we obtain 7,639 MW.

Now, let's assume that the nuclear generator can only be built in discrete steps of 1,000 MW. We can set `build_mode` to `"modular"` and the `p_nom_mod` attribute to 1,000 MW, which introduces an integer variable that constraints the optimised capacity to be a multiple of 1,000 MW. To constrain the option space for the integer variable, and help the solver a bit, we can also set the upper limit `p_nom_max` to 10,000 MW.

This problem is solved as a mixed-integer linear program (MILP), which will take longer to solve than the previous continuous linear program (LP).

In [ ]:
n.generators.loc["nuclear", "p_nom_mod"] = 1000
n.generators.loc["nuclear", "build_mode"] = "modular"
n.generators.loc["nuclear", "p_nom_max"] = 10000
n.optimize(solver_name="gurobi");

From the re-optimised results, we can see that the optimal capacity for the nuclear generator is now 8,000 MW, a multiple of 1,000 MW.

Let's compare the two results.
The first two panels show the capital expenditure of the nuclear generator at each capacity it is allowed to build (in blue), and the capacity that the optimisation chose (the black ring).
With `build_mode="continuous"`, any capacity on the line is allowed, so the model builds 7.6 GW.
With `build_mode="modular"`, only multiples of 1,000 MW (in PyPSA - a "module") are allowed, so the model builds eight modules instead.
The right panel shows that the extra nuclear capacity replaces some solar and wind capacity.

In [ ]:
nuclear = n.generators.loc["nuclear"]
capacity = np.arange(0, nuclear.p_nom_max + 1, nuclear.p_nom_mod)
results = {
    "continuous": ("-", p_nom_continuous, capex_continuous),
    "modular": (
        "o",
        n.generators.p_nom_opt,
        n.statistics.capex(components="Generator", groupby="name"),
    ),
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4), width_ratios=[1, 1, 1.3])

for ax, (build_mode, (style, p_nom, capex)) in zip(axes, results.items()):
    ax.plot(
        capacity / 1e3,
        nuclear.capital_cost * capacity / 1e9,
        style,
        color="#2a78d6",
        label="allowed",
    )
    ax.plot(
        p_nom["nuclear"] / 1e3,
        capex["nuclear"] / 1e9,
        "o",
        color="black",
        markersize=14,
        fillstyle="none",
        markeredgewidth=2,
        label="optimum",
    )
    ax.set(
        title=f'build_mode="{build_mode}"',
        xlabel="Nuclear capacity [GW]",
        ylabel="Capital expenditure [bn€/a]",
    )
    ax.legend(loc="upper left")

capacities = pd.DataFrame(
    {"continuous": p_nom_continuous, "modular": n.generators.p_nom_opt}
)
capacities = capacities.drop("load shedding")
(capacities / 1e3).plot.barh(ax=axes[2], color=["#2a78d6", "#eb6834"], width=0.7)
axes[2].set(
    xlabel="Optimal capacity [GW]",
    ylabel="",
    title="Generators other than load shedding",
)

fig.tight_layout()

As a short concluding excursion into the results, we can also illustrate how the optimal dispatch of this nuclear generator is constrained by ramp limits and minimum part loads in January 2019.
The shaded band is the range in which the generator must operate: between the minimum part load of 70% and the built capacity.
The ramp limit of 3%/h stops the dispatch from jumping between the two edges of the band from one snapshot to the next.

In [ ]:
p_nom = n.generators.at["nuclear", "p_nom_opt"] / 1e3
p_min = n.generators.at["nuclear", "p_min_pu"] * p_nom
dispatch = n.generators_t.p.loc["2019-01", "nuclear"] / 1e3

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.axhspan(p_min, p_nom, color="#2a78d6", alpha=0.12, label="allowed operating range")
ax.plot(
    dispatch.index, dispatch, color="#2a78d6", linewidth=2, label="nuclear dispatch"
)
ax.axhline(p_nom, color="0.3", linestyle="--", linewidth=1)
ax.axhline(p_min, color="0.3", linestyle=":", linewidth=1)
ax.annotate(
    f"capacity: {p_nom:.0f} GW",
    (dispatch.index[0], p_nom),
    xytext=(4, 4),
    textcoords="offset points",
)
ax.annotate(
    f"minimum part load: {p_min:.1f} GW",
    (dispatch.index[0], p_min),
    xytext=(4, -14),
    textcoords="offset points",
)
ax.set(
    ylabel="Power [GW]",
    ylim=(0, p_nom * 1.15),
    xlim=(dispatch.index[0], dispatch.index[-1]),
)
ax.legend(loc="lower right")